In [0]:
CREATE OR REPLACE TABLE workspace.default.tb_apolices

AS

SELECT      DISTINCT
            APO.id_apolice                      AS ID_APOLICE
            ,APO.id_cliente                     AS ID_CLIENTE
            ,APO.cod_prod                       AS ID_PRODUTO
            ,PRO.nome_produto                   AS DS_PRODUTO
            ,PRO.linha_negocio                  AS DS_LINHA_NEGOCIO
            ,APO.premio_bruto                   AS VL_PREMIO_BRUTO
            ,TO_DATE(APO.dt_emissao)            AS DT_EMISSAO
            ,TO_DATE(APO.dt_inicio_vigencia)    AS DT_INICIO_VIGENCIA
            ,TO_DATE(APO.dt_fim_vigencia)       AS DT_FIM_VIGENCIA
            ,APO.status_apolice                 AS DS_STATUS_APOLICE
            ,TO_DATE(APO.dt_cancelamento)       AS DT_CANCELAMENTO
            ,APO.id_apolice_anterior            AS ID_APOLICE_ANTERIOR
            ,CASE WHEN APO.status_apolice = 'CANCELADA' 
                THEN 'CANCELADA'
            WHEN APO.dt_inicio_vigencia > current_timestamp 
                THEN 'A INICIAR'
            WHEN APO.dt_fim_vigencia IS NULL 
                THEN 'VIGENTE - ABERTA'
            WHEN APO.dt_fim_vigencia >= current_timestamp 
                THEN 'VIGENTE'
                    ELSE 'EXPIRADA' END         AS DS_SITUACAO_VIGENCIA
FROM        workspace.default.tb_apolices_v2    APO
INNER JOIN  workspace.default.tb_produtos       PRO
ON          APO.cod_prod = PRO.cod_prod


num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.default.tb_parcelas

AS

WITH CALENDARIO AS (
    SELECT TIMESTAMP '2026-10-04 23:59:59' AS DT_CORTE
)

SELECT  PAR.id_apolice AS ID_APOLICE
        ,PAR.id_parcela AS ID_PARCELA
        ,PAR.num_parcela AS NR_PARCELA
        ,TO_DATE(PAR.dt_vencimento) AS DT_VENCIMENTO
        ,TO_DATE(PAR.dt_pagamento) AS DT_PAGAMENTO
        ,PAR.valor_parcela AS VL_PARCELA
        ,PAR.tipo_movimento AS DS_TIPO_MOVIMENTO
        -- , CASE
        --     WHEN PAR.tipo_movimento = 'ESTORNO' 
        --         THEN 'ESTORNO'
        --     WHEN PAR.status_parcela = 'CANCELADA' 
        --         THEN 'CANCELADA'
        --     WHEN PAR.dt_pagamento IS NOT NULL 
        --         THEN 'PAGA'
        --     WHEN PAR.dt_vencimento > CLD.DT_CORTE 
        --         THEN 'A VENCER'
        --     ELSE 'INADIMPLENTE' END DS_SITUACAO_PARCELA
        ,CASE
            WHEN PAR.tipo_movimento = 'ESTORNO' THEN 'ESTORNO'
            WHEN PAR.status_parcela = 'CANCELADA' THEN 'CANCELADA'
            WHEN PAR.dt_pagamento IS NOT NULL THEN 'PAGA'
            WHEN PAR.dt_vencimento > TIMESTAMP'2026-10-04 23:59:59' THEN 'A VENCER'
            WHEN PAR.dt_vencimento <= TIMESTAMP'2026-09-04 23:59:59' THEN 'INADIMPLENTE'
            ELSE 'EM ATRASO'
            END AS DS_SITUACAO_PARCELA
        ,CASE
            WHEN PAR.dt_pagamento IS NOT NULL AND PAR.tipo_movimento = 'PARCELA'
                THEN greatest(datediff(to_date(PAR.dt_pagamento), to_date(PAR.dt_vencimento)), 0)
            WHEN PAR.dt_pagamento IS NULL AND PAR.status_parcela = 'ABERTA'
                AND PAR.dt_vencimento <= CLD.DT_CORTE
                    THEN datediff(to_date(CLD.DT_CORTE), to_date(PAR.dt_vencimento))
            END AS DS_DIAS_ATRASO
FROM    workspace.default.f_parcelas_v2     PAR
INNER JOIN workspace.default.tb_apolices    APO
ON PAR.id_apolice = APO.ID_APOLICE
INNER JOIN CALENDARIO                       CLD

num_affected_rows,num_inserted_rows


In [0]:
SELECT      DS_SITUACAO_PARCELA, count(*) 
FROM        workspace.default.tb_parcelas
GROUP BY    DS_SITUACAO_PARCELA

DS_SITUACAO_PARCELA,count(1)
PAGA,1014491
A VENCER,194705
INADIMPLENTE,41561
EM ATRASO,9196
CANCELADA,38346
ESTORNO,7492


In [0]:
CREATE OR REPLACE TABLE workspace.default.tb_premio_emitido_final

 AS

SELECT  DATE_TRUNC('month', DT_EMISSAO) AS DT_REFERENCIA
        ,DS_LINHA_NEGOCIO
        ,DS_PRODUTO
        ,COUNT(*) AS QT_APOLICES
        ,ROUND(SUM(VL_PREMIO_BRUTO), 2) AS VL_PREMIO
        ,ROUND(AVG(VL_PREMIO_BRUTO), 2) AS VL_TKT_MEDIO
FROM    workspace.default.tb_apolices

GROUP BY ALL

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.default.tb_arrecadacao_final

AS

SELECT      DATE_TRUNC('month', PAR.DT_PAGAMENTO) AS DT_REFERENCIA
            ,APO.DS_LINHA_NEGOCIO
            ,APO.DS_PRODUTO
            ,ROUND(SUM(CASE WHEN PAR.DS_TIPO_MOVIMENTO = 'PARCELA' THEN PAR.VL_PARCELA END), 2) AS VL_ARRECADACAO_BRUTA
            ,ROUND(SUM(CASE WHEN PAR.DS_TIPO_MOVIMENTO = 'ESTORNO' THEN PAR.VL_PARCELA END), 2) AS VL_ESTORNO
            ,ROUND(SUM(PAR.VL_PARCELA), 2) AS VL_ARRECADACAO_LIQUIDA
FROM        workspace.default.tb_parcelas           PAR
INNER JOIN  workspace.default.tb_apolices           APO
ON          PAR.ID_APOLICE = APO.ID_APOLICE
WHERE       PAR.DT_PAGAMENTO IS NOT NULL
GROUP BY    ALL

num_affected_rows,num_inserted_rows


In [0]:
SELECT  ROUND(SUM(VL_ARRECADACAO_BRUTA),2)
        ,ROUND(SUM(VL_ESTORNO),2)
        ,ROUND(SUM(VL_ARRECADACAO_LIQUIDA),2)
FROM    workspace.default.arrecadacao_final

"round(sum(VL_ARRECADACAO_BRUTA), 2)","round(sum(VL_ESTORNO), 2)","round(sum(VL_ARRECADACAO_LIQUIDA), 2)"
1.3087072358E9,-1.892404266E7,1.28978319314E9


In [0]:
CREATE OR REPLACE TABLE workspace.default.tb_inadimplencia_final

AS

SELECT      DATE_TRUNC('MONTH', PAR.DT_VENCIMENTO) AS MES_VENCIMENTO
            ,APO.DS_LINHA_NEGOCIO
            ,COUNT(*) AS QTD_PARCELAS_VENCIDAS
            ,COUNT(CASE WHEN PAR.DS_SITUACAO_PARCELA = 'INADIMPLENTE' THEN 1 END) AS QTD_INADIMPLENTES
            ,ROUND(SUM(PAR.VL_PARCELA), 2) AS VALOR_VENCIDO
            ,ROUND(SUM(CASE WHEN PAR.DS_SITUACAO_PARCELA = 'INADIMPLENTE' THEN PAR.VL_PARCELA END), 2) AS VALOR_INADIMPLENTE
            ,ROUND(SUM(CASE WHEN PAR.DS_SITUACAO_PARCELA = 'INADIMPLENTE' THEN PAR.VL_PARCELA END)
            / SUM(PAR.VL_PARCELA) * 100, 2) AS TAXA_INADIMPLENCIA_PCT
FROM        workspace.default.tb_parcelas   PAR
INNER JOIN  workspace.default.tb_apolices   APO
ON          PAR.ID_APOLICE = APO.ID_APOLICE
WHERE       PAR.DS_TIPO_MOVIMENTO = 'PARCELA'
AND         PAR.DS_SITUACAO_PARCELA IN ('PAGA', 'INADIMPLENTE')
AND         TO_DATE(PAR.dt_vencimento) <= '2026-09-04'
GROUP BY 1, 2;

num_affected_rows,num_inserted_rows


In [0]:
SELECT  ROUND(SUM(VALOR_VENCIDO),2) AS VL_VENCIDO
        ,ROUND(SUM(VALOR_INADIMPLENTE),2) AS VL_INADIMPLENTE
        ,ROUND(SUM(VALOR_INADIMPLENTE) / SUM(VALOR_VENCIDO) * 100, 2) AS PCT_TAXA-- TAXA INADIMPLENCIA
FROM    workspace.default.tb_inadimplencia_final


VL_VENCIDO,VL_INADIMPLENTE,PCT_TAXA
1.30423837393E9,5.320002385E7,4.08


In [0]:
CREATE OR REPLACE TABLE workspace.default.tb_premio_ganho_final 

AS

WITH BASE AS (
 SELECT  ID_APOLICE
         ,DS_LINHA_NEGOCIO
         ,DS_PRODUTO
         ,VL_PREMIO_BRUTO
         ,TO_DATE(DT_INICIO_VIGENCIA) AS INI
         -- Data final efetiva da apólice: considera o menor valor entre a data fim de vigência
         -- (ou início + 365 dias se nula), a data de cancelamento (ou 9999-12-31 se nula) e a
         -- data de corte fixa do período de análise (2026-10-05). Determina até quando o prêmio
         -- deve ser reconhecido/earning durante o período de vigência da apólice.
         ,LEAST(
            COALESCE(TO_DATE(DT_FIM_VIGENCIA), DATE_ADD(TO_DATE(DT_INICIO_VIGENCIA), 365)),
            COALESCE(TO_DATE(DT_CANCELAMENTO), DATE'9999-12-31'),
            DATE'2026-10-05'
         ) AS FIM_EFETIVO
 FROM   workspace.default.tb_apolices
),

-- Gera uma linha para cada mês de competência entre o início da vigência  e o dia anterior ao FIM_EFETIVO com intervalo de 1 mês. Cada mês gerado representa um período dE competência sobre o qual o prêmio bruto será rateado  para cálculo do prêmio ganho.
MESES AS (
    SELECT  *
            ,EXPLODE(SEQUENCE(TRUNC(INI, 'MONTH'), TRUNC(DATE_SUB(FIM_EFETIVO, 1), 'MONTH'), INTERVAL 1 MONTH)) AS MES
    FROM    BASE
    WHERE   FIM_EFETIVO > INI
)

-- Prêmio ganho: rateia o prêmio bruto pro-rata pelo número de dias de vigência dentro de cada mês de competência (prêmio bruto / 365 dias × dias no mês).
SELECT MES AS MES_COMPETENCIA
        ,DS_LINHA_NEGOCIO
        ,DS_PRODUTO
        ,ROUND(SUM(VL_PREMIO_BRUTO / 365.0
        * DATEDIFF(LEAST(FIM_EFETIVO, add_months(MES, 1)), GREATEST(INI, MES))), 2) AS VL_PREMIO_GANHO
FROM    MESES

GROUP BY ALL 
ORDER BY 1 DESC

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.default.gold_kpis_mensal 

AS

WITH EMITIDO AS (
    SELECT  CAST(DT_REFERENCIA AS DATE) AS MES,
            DS_LINHA_NEGOCIO,
            SUM(QT_APOLICES) AS QT_APOLICES,
            SUM(VL_PREMIO) AS VL_PREMIO
    FROM    workspace.default.tb_premio_emitido_final
    GROUP BY 1, 2
),

GANHO AS (
    SELECT  CAST(MES_COMPETENCIA AS DATE) AS MES,
            DS_LINHA_NEGOCIO,
            SUM(VL_PREMIO_GANHO) AS VL_PREMIO_GANHO
    FROM    workspace.default.tb_premio_ganho_final
    GROUP BY 1, 2
),

ARREC AS (
    SELECT  CAST(DT_REFERENCIA AS DATE) AS MES,
            DS_LINHA_NEGOCIO,
            SUM(VL_ARRECADACAO_BRUTA) AS VL_ARRECADACAO_BRUTA,
            SUM(VL_ESTORNO) AS VL_ESTORNO,
            SUM(VL_ARRECADACAO_LIQUIDA) AS VL_ARRECADACAO_LIQUIDA
    FROM    workspace.default.tb_arrecadacao_final
    GROUP BY 1, 2
),

INAD AS (
    SELECT  CAST(MES_VENCIMENTO AS DATE) AS MES,
            DS_LINHA_NEGOCIO,
            SUM(VALOR_VENCIDO) AS VALOR_VENCIDO,
            SUM(VALOR_INADIMPLENTE) AS VALOR_INADIMPLENTE
    FROM    workspace.default.tb_inadimplencia_final
    GROUP BY 1, 2
),

CHAVES AS (
    SELECT MES, DS_LINHA_NEGOCIO FROM EMITIDO
    UNION SELECT MES, DS_LINHA_NEGOCIO FROM GANHO
    UNION SELECT MES, DS_LINHA_NEGOCIO FROM ARREC
    UNION SELECT MES, DS_LINHA_NEGOCIO FROM INAD
)

SELECT  C.MES,
        C.DS_LINHA_NEGOCIO,
        COALESCE(E.QT_APOLICES, 0) AS QT_APOLICES,
        COALESCE(E.VL_PREMIO, 0) AS VL_PREMIO_EMITIDO,
        COALESCE(G.VL_PREMIO_GANHO, 0) AS VL_PREMIO_GANHO,
        COALESCE(A.VL_ARRECADACAO_BRUTA, 0) AS VL_ARRECADACAO_BRUTA,
        COALESCE(A.VL_ESTORNO, 0) AS VL_ESTORNO,
        COALESCE(A.VL_ARRECADACAO_LIQUIDA, 0) AS VL_ARRECADACAO_LIQUIDA,
        COALESCE(I.VALOR_VENCIDO, 0) AS VALOR_VENCIDO,
        COALESCE(I.VALOR_INADIMPLENTE, 0) AS VALOR_INADIMPLENTE,
        ROUND(I.VALOR_INADIMPLENTE / NULLIF(I.VALOR_VENCIDO, 0) * 100, 2) AS TAXA_INADIMPLENCIA_PCT,
        ROUND(E.VL_PREMIO / NULLIF(E.QT_APOLICES, 0), 2) AS VL_TKT_MEDIO
FROM    CHAVES C
LEFT JOIN EMITIDO E  
ON C.MES = E.MES AND C.DS_LINHA_NEGOCIO = E.DS_LINHA_NEGOCIO
LEFT JOIN GANHO G    
ON C.MES = G.MES AND C.DS_LINHA_NEGOCIO = G.DS_LINHA_NEGOCIO
LEFT JOIN ARREC A    
ON C.MES = A.MES AND C.DS_LINHA_NEGOCIO = A.DS_LINHA_NEGOCIO
LEFT JOIN INAD I     
ON C.MES = I.MES AND C.DS_LINHA_NEGOCIO = I.DS_LINHA_NEGOCIO;

num_affected_rows,num_inserted_rows


In [0]:
SELECT round(sum(VL_ARRECADACAO_LIQUIDA),2)
        ,round(sum(VL_PREMIO_GANHO),2)
        ,round(sum(VL_PREMIO_EMITIDO),2)
 FROM workspace.default.gold_kpis_mensal

"round(sum(VL_ARRECADACAO_LIQUIDA), 2)","round(sum(VL_PREMIO_GANHO), 2)","round(sum(VL_PREMIO_EMITIDO), 2)"
1.28978319314E9,1.09962026606E9,1.58703004569E9


In [0]:
SELECT * FROM workspace.default.gold_kpis_mensal